# Flood early warning for Sri Lanka — three model families

Trains and evaluates **model 1 (TF-STGNN)**, the graph network of
`docs/PROJECT_PROPOSAL.md` §7.7; **model 2 (MMF-Net)**, a graph-free multimodal
transformer; and **model 3 (STG-Former)**, which puts model 2's input layer on
model 1's river graph so that RQ1 can be answered at all — plus the four
baselines of §7.8 and every build ladder.

## Before you hit *Save & Run All*

In the right-hand panel:

1. **Add Input** → attach:
   - `uom230429e/sri-lanka-flood-tabular-graph-2003-2025` — **required**
   - `uom230429e/flood-data-set` — only for `sar_pretrain`, `sar`, `sar2`.
     Model 3's ladder needs no imagery, so skip it for the run below.
2. **Accelerator** → **GPU T4 ×2** (or P100)
3. **Internet** → **On** — needed to clone the code

## What this session does

`--stage all` runs cheapest-and-most-decisive first, so a session that runs out
of time has still produced what matters. Anything that will not fit in
`--time-budget-hours` is skipped with the command to finish it later, and a
stage that crashes does not take the others down with it.

| Stage | Answers | Rough cost |
|---|---|---|
| `baselines` | the bar to clear | ~10 min |
| `ladder3` (P0→P3) | **RQ1** — the open question | ~4 h |
| `onset` (P4 pair) | early warning on the onset head | ~3 h |
| `ladder2` (N0→N5) | RQ6, RQ7, the tree gap | 2–3 h |
| `ladder` (M0→M5) | RQ3, RQ4 | 2–4 h |
| `leakage` | **RQ2** | ~30 min |
| `spatial` | spatial generalisation | ~30 min |
| `sar2` / `sar` | RQ5, RQ8 | 3–5 h each |

Models 1 and 2 have both already been run, so an 8 h budget spent on
`baselines → ladder3 → onset` is the whole value of this session.

## Read P0 before anything else

`P0` is an **assembly check, not a result**. It is byte-identical to model 2's
`N3` by construction, so it must land near **PR-AUC 0.8269**. If it does not,
the wiring is wrong and P1–P3 are not worth reading. It runs first and takes
about 20 minutes.


In [ ]:
# --- get the code -----------------------------------------------------
# rmtree rather than !rm -rf: this runs in the notebook process, so a re-run
# always starts from a clean checkout instead of failing on an existing dir.
import shutil, subprocess, sys

REPO = "https://github.com/heshannethmina/Srilanka-Flood-Data-Set-Creation"
DEST = "/kaggle/working/repo"

shutil.rmtree(DEST, ignore_errors=True)
subprocess.run(["git", "clone", "-q", REPO, DEST], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)
!ls {DEST}/model

In [ ]:
# --- environment check ------------------------------------------------
# Fails loudly *now* rather than three hours into the ladder.
import glob, os, torch

print(f"torch {torch.__version__} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Accelerator'}")

for name, need in [("flood_dataset.parquet", True), ("image_dataset.csv", False)]:
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    print(f"{name:24s} {'OK  ' + os.path.dirname(hits[0]) if hits else 'MISSING'}")
    assert hits or not need, f"attach the dataset containing {name} via 'Add Input'"

In [ ]:
# --- run the session --------------------------------------------------
# baselines -> ladder3 -> onset fills an 8 h budget and answers RQ1.
# Everything after that is already-published ladders and is skipped.
!python /kaggle/working/repo/models/kaggle_run.py --stage all --time-budget-hours 8

# Model 1's missing BCE rung (~40 min). Not part of --stage all, because it is
# a rung of model 1's ladder rather than a stage. Without it every M-vs-N
# comparison is confounded by the focal-loss defect, so run it here if the
# budget above left room, or in the second session.
# !python /kaggle/working/repo/models/kaggle_run.py #     --stage ladder --presets M5_bce --ladder-seeds 5

# Just model 3's ladder, if that is all you need (~4 h):
# !python /kaggle/working/repo/models/kaggle_run.py --stage ladder3


In [ ]:
# --- package the results ----------------------------------------------
# /kaggle/working is the notebook's output, so this survives 'Save Version'.
import glob, json, os

files = sorted(glob.glob("/kaggle/working/runs/*"))
print(f"{len(files)} result files:")
for f in files:
    print(f"  {os.path.basename(f):40s} {os.path.getsize(f) / 1e3:8.1f} kB")

if files:
    !cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip

## Second session

A fresh notebook with the same inputs, cells 1–2, then whichever of these the
first session skipped:

```python
# the missing BCE rung — do this one first, it is cheap and it unconfounds
# every model 1 vs model 2 comparison in the paper
!python /kaggle/working/repo/models/kaggle_run.py     --stage ladder --presets M5_bce --ladder-seeds 5

!python /kaggle/working/repo/models/kaggle_run.py --stage onset
```

The SAR stages (`sar_pretrain` → `sar2` → `sar`) need 3–5 h each and both SAR
rungs have already returned nulls. `sar_pretrain` must come before `sar2`: it
writes the encoder `N6_gated` transfers, and its validation AP near the 40%
chip base rate means the encoder learned nothing visual.

## Then, offline

Download `runs.zip` from the notebook output, unzip it **into the same folder as
the earlier runs** — the paired confidence intervals need both models'
`_preds.npz` side by side — and generate the results document:

```bash
python models/results_doc.py runs/ --out docs/RESULTS.md
```
